<a href="https://colab.research.google.com/github/avalee0215/COMPSYS-306_Project2/blob/ava_project1/COMPSYS306_clee482_SVM_only.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Google Drive mount
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [ ]:
# Data loading and train/val/test split
import os, json, sys
import numpy as np
from sklearn.model_selection import train_test_split

# ---------- Config ----------
ARCHIVE_DIR   = "/content/gdrive/MyDrive/CS306_2025/database/archive"
PROCESSED_DIR = os.path.join(ARCHIVE_DIR, "processed")

STEP1_NPZ = os.path.join(PROCESSED_DIR, "arrays_flatten_32x32x3.npz")
OUT_SPLITS = os.path.join(PROCESSED_DIR, "splits_flatten_32x32x3.npz")

RANDOM_STATE = 42
TEST_SIZE = 0.20   # 20% for test
VAL_SIZE  = 0.10   # 10% of train portion

def main():
    print("2): Stratified Train/Val/Test Split")

    # Safety checks
    if not os.path.exists(STEP1_NPZ):
        print(f"[Error] Step 1 cache not found: {STEP1_NPZ}", file=sys.stderr)
        sys.exit(1)

    # Load cached arrays from Step 1
    data = np.load(STEP1_NPZ)
    X, y = data["X"], data["y"]
    print(f"[Info] Loaded: X={X.shape}, y={y.shape}")

    # First split -> Train/Test (stratified)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_STATE,
        shuffle=True
    )
    print(f"[Split 1] Train={X_train.shape[0]}, Test={X_test.shape[0]}")

    # Second split -> Train/Val (stratified, from the train portion)
    # Val fraction relative to the *current* train size:
    val_size_abs = int(np.floor(VAL_SIZE * X_train.shape[0]))
    # Convert to fraction for train_test_split (must be in (0,1))
    val_frac = val_size_abs / X_train.shape[0] if X_train.shape[0] > 0 else 0.1

    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train,
        test_size=val_frac,
        stratify=y_train,
        random_state=RANDOM_STATE,
        shuffle=True
    )
    print(f"[Split 2] Train={X_train.shape[0]}, Val={X_val.shape[0]} (≈{VAL_SIZE*100:.0f}% of train)")


    def cls_counts(y_arr):
        cls, cnt = np.unique(y_arr, return_counts=True)
        return dict(zip(cls.tolist(), cnt.tolist()))
    print(f"[Check] Class counts (train) sample: {str(dict(list(cls_counts(y_train).items())[:5]))} ...")


    np.savez_compressed(
        OUT_SPLITS,
        X_train=X_train, y_train=y_train,
        X_val=X_val,     y_val=y_val,
        X_test=X_test,   y_test=y_test
    )
    print(f"[Saved] {OUT_SPLITS}")

if __name__ == "__main__":
    main()


In [ ]:
# Data loading and train/val/test split
# PCA 2D scatter of all splits
import os, numpy as np, matplotlib.pyplot as plt
from sklearn.decomposition import PCA

ARCHIVE_DIR = next((p for p in [
    "/content/drive/MyDrive/CS306_2025/database/archive",
    "/content/gdrive/MyDrive/CS306_2025/database/archive",
] if os.path.exists(p)), "/content/drive/MyDrive/CS306_2025/database/archive")
PROC   = os.path.join(ARCHIVE_DIR, "processed")
FIGS   = os.path.join(ARCHIVE_DIR, "report", "figures")
os.makedirs(FIGS, exist_ok=True)

d = np.load(os.path.join(PROC, "splits_flatten_32x32x3.npz"))
X_all = np.vstack([d["X_train"], d["X_val"], d["X_test"]])
y_all = np.hstack([d["y_train"], d["y_val"], d["y_test"]])


rng = np.random.RandomState(42)
if X_all.shape[0] > 8000:
    idx = rng.choice(X_all.shape[0], size=8000, replace=False)
    X_sub, y_sub = X_all[idx], y_all[idx]
else:
    X_sub, y_sub = X_all, y_all

Z = PCA(n_components=2, random_state=42).fit_transform(X_sub)

plt.figure(figsize=(6.5, 5.5))
plt.scatter(Z[:,0], Z[:,1], c=y_sub, s=4, alpha=0.6, cmap="tab20")
plt.title("Traffic Signs — PCA (2D projection)")
plt.tight_layout()
plt.savefig(os.path.join(FIGS, "pca2_scatter.png"), dpi=200)
plt.close()
print("[Saved] pca2_scatter.png")


SVM

Compare: linear-SVM, RBF(no pca), RBF(pca:100)

In [ ]:
# Data loading and train/val/test split
# SVM FINAL REPORT — linear

import os, sys, json, pickle, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (classification_report, accuracy_score, f1_score,
                             precision_score, recall_score, confusion_matrix,
                             ConfusionMatrixDisplay)
from sklearn.model_selection import train_test_split


ARCHIVE_DIR = next((p for p in [
    "/content/drive/MyDrive/CS306_2025/database/archive",
    "/content/gdrive/MyDrive/CS306_2025/database/archive",
] if os.path.exists(p)), "/content/drive/MyDrive/CS306_2025/database/archive")
PROC   = os.path.join(ARCHIVE_DIR, "processed")
MODELS = os.path.join(ARCHIVE_DIR, "models")
FIGS   = os.path.join(ARCHIVE_DIR, "report", "figures")
os.makedirs(FIGS, exist_ok=True)


def load_splits(proc_dir, seed=42):
    # Try splits_* first
    for fname in ["splits_flatten_32x32x3.npz", "splits_32x32_flatten.npz", "splits.npz"]:
        p = os.path.join(proc_dir, fname)
        if not os.path.exists(p):
            continue
        d = np.load(p)
        keys = set(d.files)
        candidates = [
            ("Xtr","Xval","Xte","ytr","yval","yte"),
            ("X_train","X_val","X_test","y_train","y_val","y_test"),
            ("Xtrain","Xval","Xtest","Ytrain","Yval","Ytest"),
            ("XTr","XVal","XTe","yTr","yVal","yTe"),
        ]
        for XtrK, XvalK, XteK, ytrK, yvalK, yteK in candidates:
            if {XtrK,XvalK,XteK,ytrK,yvalK,yteK}.issubset(keys):
                return d[XtrK], d[ytrK], d[XvalK], d[yvalK], d[XteK], d[yteK]
    # Fallback: arrays_* (X,y) -> make 72/8/20
    for fname in ["arrays_flatten_32x32x3.npz", "arrays_32x32_flatten.npz", "arrays.npz"]:
        p = os.path.join(proc_dir, fname)
        if not os.path.exists(p):
            continue
        d = np.load(p)
        keys = set(d.files)
        Xkey = next((k for k in ["X","Xall","X_all","features"] if k in keys), None)
        ykey = next((k for k in ["y","yall","y_all","labels"] if k in keys), None)
        if Xkey and ykey:
            X, y = d[Xkey], d[ykey]
            X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, stratify=y, random_state=seed)
            X_tr, X_val, y_tr, y_val = train_test_split(X_tr, y_tr, test_size=0.10, stratify=y_tr, random_state=seed)
            return X_tr, y_tr, X_val, y_val, X_te, y_te
    raise FileNotFoundError("Could not load splits from processed/. Provide splits_* or arrays_* npz.")


def load_label_names(proc_dir):
    for fname in ["label_map_flat.json", "label_map.json"]:
        p = os.path.join(proc_dir, fname)
        if os.path.exists(p):
            with open(p, "r", encoding="utf-8") as f:
                lm = json.load(f)
            if isinstance(lm, dict) and "idx_to_name" in lm:
                lm = lm["idx_to_name"]
            return {str(k): v for k, v in lm.items()}
    return None


def save_confmats(y_true, y_pred, labels_sorted, display_names, tag):
    for norm_tag, suffix in [(None, "raw"), ("true", "norm")]:
        cm = confusion_matrix(y_true, y_pred, labels=labels_sorted, normalize=norm_tag)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=display_names if display_names is not None else labels_sorted)
        fig, ax = plt.subplots(figsize=(8,6))
        disp.plot(ax=ax, colorbar=False, cmap="Blues", xticks_rotation=90)
        ax.set_title(f"{tag} — Confusion Matrix (normalize={norm_tag})")
        plt.tight_layout()
        plt.savefig(os.path.join(FIGS, f"confmat_{tag}_{suffix}.png"), dpi=170)
        plt.close(fig)

def save_perclass_f1(y_true, y_pred, class_names, tag):
    rep = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
    df = pd.DataFrame(rep).T.iloc[:-3]  # drop macro/weighted/accuracy rows
    plt.figure(figsize=(max(12, len(df)*0.28), 4.5))
    df["f1-score"].plot(kind="bar")
    plt.ylim(0, 1.0); plt.ylabel("F1-score"); plt.title(f"{tag} — per-class F1 (TEST)")
    plt.tight_layout(); plt.savefig(os.path.join(FIGS, f"f1_per_class_{tag}.png"), dpi=170); plt.close()


def eval_and_export(tag, clf, Xte_raw, yte, scaler_for_linear=None, label_map=None):

    if tag.startswith("svm_linear") and scaler_for_linear is not None:
        X_in = scaler_for_linear.transform(Xte_raw)
    else:
        X_in = Xte_raw  # pipeline will scale if needed

    t0 = time.time()
    yhat = clf.predict(X_in)
    infer_time = time.time() - t0

    # Metrics
    acc = accuracy_score(yte, yhat)
    f1m = f1_score(yte, yhat, average="macro")
    pm  = precision_score(yte, yhat, average="macro", zero_division=0)
    rm  = recall_score(yte, yhat, average="macro", zero_division=0)

    # Full report CSV
    if label_map:
        class_names = [label_map.get(str(i), str(i)) for i in range(len(np.unique(yte)))]
        rep = classification_report(yte, yhat, target_names=class_names, digits=3, output_dict=True)
    else:
        rep = classification_report(yte, yhat, digits=3, output_dict=True)

    pd.DataFrame(rep).T.to_csv(os.path.join(FIGS, f"{tag}_report_test.csv"))

    with open(os.path.join(FIGS, f"{tag}_metrics.json"), "w") as f:
        json.dump({
            "model": tag,
            "test": {
                "accuracy": float(acc),
                "macro_f1": float(f1m),
                "macro_precision": float(pm),
                "macro_recall": float(rm),
                "infer_time_sec": float(infer_time)
            }
        }, f, indent=2)

    # Confusion matrices & per-class F1
    labels_sorted = np.unique(yte)
    display_names = None
    if label_map:
        display_names = [label_map.get(str(i), str(i)) for i in labels_sorted]
    save_confmats(yte, yhat, labels_sorted, display_names, tag)
    if label_map:
        class_names_full = [label_map.get(str(i), str(i)) for i in range(len(label_map))]
        save_perclass_f1(yte, yhat, class_names_full, tag)

    return {
        "model": tag,
        "test_accuracy": acc,
        "test_macro_f1": f1m,
        "test_macro_precision": pm,
        "test_macro_recall": rm,
        "infer_time_sec": infer_time
    }

# ===================== MAIN =====================
# Load data
Xtr, ytr, Xval, yval, Xte, yte = load_splits(PROC)
label_map = load_label_names(PROC)

# Load models
paths = {
    "svm_linear_final": os.path.join(MODELS, "svm_final.pkl"),
    "svm_rbf_c10":      os.path.join(MODELS, "svm_rbf_c10.pkl"),
    "svm_rbf_c10_pca100": os.path.join(MODELS, "svm_rbf_c10_pca100.pkl"),
}
models = {}
for tag, p in paths.items():
    if os.path.exists(p):
        with open(p, "rb") as f:
            models[tag] = pickle.load(f)
    else:
        print(f"[Info] skip (missing): {p}")

if "svm_linear_final" not in models:
    print("[Error] Missing linear final model: models/svm_final.pkl", file=sys.stderr)
    sys.exit(1)

# Load external scaler for linear
scaler_path = os.path.join(PROC, "svm_scaler.pkl")
scaler = None
if os.path.exists(scaler_path):
    with open(scaler_path, "rb") as f:
        scaler = pickle.load(f)
else:
    print("[Warn] svm_scaler.pkl not found. Linear SVM will be evaluated on raw inputs (accuracy may drop).")

# Evaluate all loaded SVM models on TEST
rows = []
for tag, clf in models.items():
    summary = eval_and_export(tag, clf, Xte, yte, scaler_for_linear=scaler, label_map=label_map)
    rows.append(summary)
    print(f"[Test] {tag:18s} acc={summary['test_accuracy']:.4f}  "
          f"F1={summary['test_macro_f1']:.4f}  P={summary['test_macro_precision']:.4f}  "
          f"R={summary['test_macro_recall']:.4f}  time={summary['infer_time_sec']:.2f}s")

# Save comparison CSV
comp_csv = os.path.join(FIGS, "svm_models_test_metrics.csv")
pd.DataFrame(rows).to_csv(comp_csv, index=False)
print(f"[Saved] {comp_csv}")
print("[Done] Reports, metrics, confusion matrices, and per-class F1 for SVMs.")


[Test] svm_linear_final   acc=0.9760  F1=0.9751  P=0.9745  R=0.9761  time=772.57s
[Test] svm_rbf_c10        acc=0.9849  F1=0.9846  P=0.9893  R=0.9804  time=1966.61s
[Test] svm_rbf_c10_pca100 acc=0.9763  F1=0.9777  P=0.9836  R=0.9724  time=105.84s
[Saved] /content/gdrive/MyDrive/CS306_2025/database/archive/report/figures/svm_models_test_metrics.csv
[Done] Reports, metrics, confusion matrices, and per-class F1 for SVMs.


In [ ]:
# Data loading and train/val/test split
# Build splits_flatten_32x32x3.npz from arrays_flatten_32x32x3.npz
import os, json, numpy as np
from sklearn.model_selection import train_test_split

ARCHIVE_DIR = "/content/gdrive/MyDrive/CS306_2025/database/archive"
PROC = os.path.join(ARCHIVE_DIR, "processed")
arrays_p = os.path.join(PROC, "arrays_flatten_32x32x3.npz")
splits_p = os.path.join(PROC, "splits_flatten_32x32x3.npz")
seed = 42

d = np.load(arrays_p)
X, y = d["X"], d["y"]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, stratify=y, random_state=seed)
X_tr, X_val, y_tr, y_val = train_test_split(X_tr, y_tr, test_size=0.10, stratify=y_tr, random_state=seed)

np.savez_compressed(splits_p, Xtr=X_tr, ytr=y_tr, Xval=X_val, yval=y_val, Xte=X_te, yte=y_te)
print("[Saved]", splits_p)


[Saved] /content/gdrive/MyDrive/CS306_2025/database/archive/processed/splits_flatten_32x32x3.npz


Comparing the models

 Comparison

In [ ]:
# Data loading and train/val/test split
# === Paths & data splits (re-usable) ===
import os, numpy as np
from sklearn.model_selection import train_test_split

ARCHIVE_DIR = next((p for p in [
    "/content/drive/MyDrive/CS306_2025/database/archive",
    "/content/gdrive/MyDrive/CS306_2025/database/archive",
    "/content/gdrive/MyDrive/CS306_2025/database/archive/"
] if os.path.exists(p)), "/content/gdrive/MyDrive/CS306_2025/database/archive")

PROC = os.path.join(ARCHIVE_DIR, "processed")
arrays_p = os.path.join(PROC, "arrays_flatten_32x32x3.npz")
splits_p = os.path.join(PROC, "splits_flatten_32x32x3.npz")
seed = 42

if os.path.exists(splits_p):
    d = np.load(splits_p)
    Xtr, ytr = d["Xtr"], d["ytr"]
    Xval, yval = d["Xval"], d["yval"]
    Xte,  yte  = d["Xte"],  d["yte"]
    print("[Loaded splits]", splits_p, Xtr.shape, Xval.shape, Xte.shape)
else:
    d = np.load(arrays_p)
    X, y = d["X"], d["y"]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.20, stratify=y, random_state=seed)
    Xtr, Xval, ytr, yval = train_test_split(Xtr, ytr, test_size=0.10, stratify=ytr, random_state=seed)
    np.savez_compressed(splits_p, Xtr=Xtr, ytr=ytr, Xval=Xval, yval=yval, Xte=Xte, yte=yte)
    print("[Saved splits]", splits_p)


In [ ]:
# Train & evaluate Linear SVM (with StandardScaler)
# === Linear SVM (with StandardScaler) ===
import time, pickle
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

lin_pipe = Pipeline([
    ("scaler", StandardScaler(with_mean=True, with_std=True)),
    ("clf", LinearSVC(C=1.0, loss="hinge", random_state=0))
])

t0 = time.time()
lin_pipe.fit(Xtr, ytr)
t1 = time.time()

def eval_split(name, X, y):
    yp = lin_pipe.predict(X)
    acc = accuracy_score(y, yp)
    f1m = f1_score(y, yp, average="macro")
    print(f"[LinearSVM] {name:>4} acc={acc:.4f} macroF1={f1m:.4f}")
    return yp

_ = eval_split("VAL", Xval, yval)
_ = eval_split("TEST", Xte,  yte)

# Save model (optional)
MODELS = os.path.join(ARCHIVE_DIR, "models")
os.makedirs(MODELS, exist_ok=True)
with open(os.path.join(MODELS, "svm_linear.pkl"), "wb") as f:
    pickle.dump(lin_pipe, f)
print("[Saved]", os.path.join(MODELS, "svm_linear.pkl"), "train_time_sec=", round(t1 - t0, 3))


In [ ]:
# Tune/Train RBF SVM (with StandardScaler)
# === RBF SVM (StandardScaler + SVC) with quick val tuning ===
import time, pickle, itertools
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

param_grid = {
    "C":    [1, 3, 10],
    "gamma":["scale", 0.01, 0.001]
}
best = {"acc": -1, "cfg": None, "model": None}

for C in param_grid["C"]:
    for gamma in param_grid["gamma"]:
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(kernel="rbf", C=C, gamma=gamma))
        ])
        t0 = time.time()
        pipe.fit(Xtr, ytr)
        val_pred = pipe.predict(Xval)
        acc = accuracy_score(yval, val_pred)
        f1m = f1_score(yval, val_pred, average="macro")
        dt = time.time() - t0
        print(f"[RBF] C={C:<4} gamma={str(gamma):<7} VAL acc={acc:.4f} macroF1={f1m:.4f} (fit {dt:.2f}s)")
        if acc > best["acc"]:
            best = {"acc": acc, "cfg": (C, gamma), "model": pipe}

Cbest, gbest = best["cfg"]
print(f"[RBF] Best on VAL: C={Cbest}, gamma={gbest}, acc={best['acc']:.4f}")

# Test with best
from sklearn.metrics import classification_report
ypred = best["model"].predict(Xte)
print("[RBF] TEST report\n", classification_report(yte, ypred, digits=4))

MODELS = os.path.join(ARCHIVE_DIR, "models")
os.makedirs(MODELS, exist_ok=True)
import pickle
with open(os.path.join(MODELS, "svm_rbf.pkl"), "wb") as f:
    pickle.dump(best["model"], f)
print("[Saved]", os.path.join(MODELS, "svm_rbf.pkl"))
